In [ ]:
import nltk
import re
from collections import Counter
import spacy

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

nlp = spacy.load("en_core_web_sm")

text = """
I recently bought a Samsung Galaxy phone from Amazon. The delivery was fast,
and the phone's camera quality is amazing! However, the battery life could be better.
Overall, I am satisfied with the product.
"""

print("\nOriginal Text:\n", text)

lower_text = text.lower()
print("\n1. Lowercase Text:\n", lower_text)

clean_text = re.sub(r'[^a-zA-Z\s]', '', lower_text)
print("\n2. Cleaned Text:\n", clean_text)

tokens = word_tokenize(clean_text)
print("\n3. Tokens:\n", tokens)

stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in tokens if word not in stop_words]
print("\n4. Tokens after Stop Word Removal:\n", filtered_tokens)

stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in filtered_tokens]
print("\n5. Stemmed Words:\n", stemmed_words)

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()
pos_tagged_tokens = pos_tag(filtered_tokens)

lemmatized_words = [
    lemmatizer.lemmatize(word, get_wordnet_pos(tag))
    for word, tag in pos_tagged_tokens
]

print("\n6. Lemmatized Words:\n", lemmatized_words)

pos_tags = pos_tag(lemmatized_words)

print("\n7. POS Tags (after lemmatization):")
for word, tag in pos_tags:
    print(f"{word} --> {tag}")

word_freq = Counter(lemmatized_words)

print("\n8. Word Frequency:")
for word, freq in word_freq.items():
    print(f"{word}: {freq}")

doc = nlp(text)

print("\n9. Named Entities:")
for entity in doc.ents:
    print(f"{entity.text} --> {entity.label_}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.



Original Text:
 
I recently bought a Samsung Galaxy phone from Amazon. The delivery was fast,
and the phone's camera quality is amazing! However, the battery life could be better.
Overall, I am satisfied with the product.


1. Lowercase Text:
 
i recently bought a samsung galaxy phone from amazon. the delivery was fast,
and the phone's camera quality is amazing! however, the battery life could be better.
overall, i am satisfied with the product.


2. Cleaned Text:
 
i recently bought a samsung galaxy phone from amazon the delivery was fast
and the phones camera quality is amazing however the battery life could be better
overall i am satisfied with the product


3. Tokens:
 ['i', 'recently', 'bought', 'a', 'samsung', 'galaxy', 'phone', 'from', 'amazon', 'the', 'delivery', 'was', 'fast', 'and', 'the', 'phones', 'camera', 'quality', 'is', 'amazing', 'however', 'the', 'battery', 'life', 'could', 'be', 'better', 'overall', 'i', 'am', 'satisfied', 'with', 'the', 'product']

4. Tokens after 

# Bag of words:
1. Tokenize each document in words.
2. Create a vocabulary of unique words.
3. Count the occurrences of each word in every document.
4. Store the count in a document em matrix.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import re

def preprocess_document(doc_text):
    lower_text = doc_text.lower()
    clean_text = re.sub(r'[^a-zA-Z\s]', '', lower_text)
    tokens = word_tokenize(clean_text)
    filtered_tokens = [word for word in tokens if word not in stop_words]

    pos_tagged_tokens = pos_tag(filtered_tokens)

    lemmatized_words_list = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tagged_tokens
    ]
    return " ".join(lemmatized_words_list)

corpus = [
    "I recently bought a Samsung Galaxy phone from Amazon. The delivery was fast.",
    "The phone's camera quality is amazing! However, the battery life could be better.",
    "Overall, I am satisfied with the product. I love this Samsung phone."
]

print("--- Bag of Words (BoW) Model ---")

preprocessed_corpus = [preprocess_document(doc) for doc in corpus]
print("\nPreprocessed Corpus (Lemmatized):")
for i, doc in enumerate(preprocessed_corpus):
    print(f"Doc {i+1}: {doc}")

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(preprocessed_corpus)

feature_names = vectorizer.get_feature_names_out()

print("\nVocabulary (Unique Words):")
print(feature_names)

print("\nDocument-Term Matrix (Bag of Words Representation):")

df_bow = pd.DataFrame(X.toarray(), columns=feature_names, index=[f'Doc {i+1}' for i in range(len(corpus))])
print(df_bow)


--- Bag of Words (BoW) Model ---

Preprocessed Corpus (Lemmatized):
Doc 1: recently buy samsung galaxy phone amazon delivery fast
Doc 2: phone camera quality amaze however battery life could better
Doc 3: overall satisfied product love samsung phone

Vocabulary (Unique Words):
['amaze' 'amazon' 'battery' 'better' 'buy' 'camera' 'could' 'delivery'
 'fast' 'galaxy' 'however' 'life' 'love' 'overall' 'phone' 'product'
 'quality' 'recently' 'samsung' 'satisfied']

Document-Term Matrix (Bag of Words Representation):
       amaze  amazon  battery  better  buy  camera  could  delivery  fast  \
Doc 1      0       1        0       0    1       0      0         1     1   
Doc 2      1       0        1       1    0       1      1         0     0   
Doc 3      0       0        0       0    0       0      0         0     0   

       galaxy  however  life  love  overall  phone  product  quality  \
Doc 1       1        0     0     0        0      1        0        0   
Doc 2       0        1     1   